# Introduction to Machine Learning
## ColoMARK PhD Network &mdash; 2.5-hour hands-on workshop

**Workshop on Mendelian Randomisation & Machine Learning methods**
School of Biomolecular and Biomedical Science (SBBS), University College Dublin
6 November 2026

Session lead: **Maryam Abbasi**, Assistant Researcher, CIAC-PLDIS, Polytechnic University of Santarem

---

### What this session is

One small machine-learning workflow, built end to end, on one dataset, and then
pulled apart to see where it could mislead you. It is deliberately **not** a tour
of algorithms. By the end you should be able to read an ML paper in your own
field and say what is missing from it.

### What you need

- A laptop (not a tablet) and a browser
- A Google account, so you can open this notebook in **Google Colab**
- Nothing to install. Every package used here is pre-installed in Colab.

### How it works

Short explanation, then something to do. Every 5&ndash;10 minutes there is a
**CHAT CHECK-IN** or an **ACTIVITY**. You are never asked to write code from
scratch; the cells run as given, and the parts you change are marked `# TODO`.

If a cell gives you red text, paste the **last line** of it in the chat and we
will sort it out.

### Plan (150 minutes)

| # | Part | Time |
|---|------|------|
| 1 | Introduction &mdash; what ML can and cannot answer | 15 min |
| 2 | Research question and data | 20 min |
| 3 | Hands-on workflow | 35 min |
| 4 | Small exercise | 10 min |
| 5 | Model evaluation | 30 min |
| 6 | Common pitfalls | 25 min |
| 7 | Group exercise and discussion | 15 min |
| | **Total** | **150 min** |

---

> **About the data.** Everything below runs on a **synthetic** colorectal
> liquid-biopsy dataset generated inside this notebook. It is built to behave
> like real data for teaching purposes &mdash; missing values, class imbalance, a
> leaky variable &mdash; but it contains **no real patients and no validated
> biomarkers**. No number produced in this notebook is clinical evidence, and
> nothing here supports any diagnostic or treatment decision.

---
## Setup &mdash; run this once

This cell imports everything the notebook needs and fixes the random seed so
your numbers match the ones on screen. Nothing to change.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 42          # fixed seed: everyone gets the same numbers
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print()
print("Setup OK.")

**Tested with:** Python 3.11&ndash;3.13, numpy &ge; 1.26, pandas &ge; 2.0,
matplotlib &ge; 3.7, scikit-learn &ge; 1.3. Google Colab's defaults satisfy all of
these. If your versions differ slightly the code still runs; only the third
decimal place of some numbers may move.

No internet access is needed at any point.

---
## Python quick reference (optional &mdash; not part of the 150 minutes)

This is a lookup table, not a lesson. Skim it now, come back to it when a line
of code looks unfamiliar. If you have never written Python, reading this before
the session is the single most useful 10 minutes of preparation.

| What you want | How it is written |
|---|---|
| Store a value | `vaf = 0.012` |
| Store text | `label = "case"` |
| A list of things | `features = ["age", "cea", "crp"]` |
| First / last item | `features[0]` / `features[-1]` (Python counts from 0) |
| How many items | `len(features)` |
| Do something to each item | `for f in features:` then an **indented** line |
| Call a ready-made function | `np.mean(x)` &mdash; name, round brackets, input |
| A table (DataFrame) | `df = pd.DataFrame(...)` |
| First 5 rows | `df.head()` |
| Rows and columns | `df.shape` |
| One column | `df["cea_ng_ml"]` |
| Several columns | `df[["age", "cea_ng_ml"]]` |
| Count each value | `df["crc"].value_counts()` |
| Count missing | `df.isna().sum()` |
| A comment (ignored) | `# anything after a hash` |

Three habits that prevent most beginner errors:

1. **Run cells in order, top to bottom.** A cell that worked yesterday will fail
   today if the cells above it have not been run in this session.
2. **Indentation is syntax, not style.** The indented lines under a `for` or an
   `if` are what gets repeated or conditioned.
3. **Read the last line of an error first.** That is where the actual problem is
   named; everything above it is the route the computer took to get there.

In [ ]:
# Optional warm-up. Nothing here is needed later - run it if Python is new to you.
vaf_values = [0.004, 0.019, 0.000, 0.071]

print("number of samples:", len(vaf_values))
print("highest VAF      :", max(vaf_values))
print("mean VAF         :", np.mean(vaf_values).round(4))

for i, v in enumerate(vaf_values):
    print(f"  sample {i}: VAF = {v:.3f}")

---
---
# PART 1 &mdash; Introduction
### 0:00 &ndash; 0:15

What machine learning can and cannot answer in cancer research.

## 1.1 Two different questions

Almost every confused ML paper in clinical research comes from mixing these up.

| | **Prediction** | **Explanation / inference** |
|---|---|---|
| Question | What is likely to be true for *this* person, given what I can measure now? | Does this variable *cause* or *influence* the outcome? |
| Success looks like | Accurate on **new, unseen** people | An unbiased, interpretable effect estimate |
| Typical tool | Classifier, risk model | Regression with a causal design, RCT, Mendelian randomisation |
| Variables are chosen for | Being measurable at the right time and adding predictive signal | Their role in a causal structure (confounder, mediator, collider) |
| A strong result means | The model may be useful in a defined setting | The variable may be a target or a mechanism |
| A strong result does **not** mean | Anything about mechanism | Anything about individual-level predictive usefulness |

**The thing to remember:** a model that predicts an outcome well tells you the
measured variables carry *information* about it. That information can come from
a cause, a consequence, a marker of care pathway, a batch effect, or an artefact
of how the cohort was assembled. The model cannot tell the difference and does
not try to.

> This is the point where today's session and this morning's Mendelian
> randomisation session separate. MR is built to support a causal claim and pays
> for it with explicit, checkable assumptions. A classifier makes no causal claim
> and offers no such assumptions. Neither is a substitute for the other.

## 1.2 What machine learning can do here

- **Classify** a sample into groups defined in advance (cancer / no cancer,
  responder / non-responder) using many measurements at once
- **Rank** people by estimated risk, so a cut-off can be chosen for a specific
  clinical use
- **Combine** many weak signals &mdash; which is exactly the liquid-biopsy situation,
  where no single marker is decisive
- **Find structure** without labels (clustering, dimension reduction) &mdash; useful
  for generating hypotheses, not for confirming them

## 1.3 What it cannot do

- **Establish causality.** A high ROC-AUC is not evidence of mechanism.
- **Rescue a badly defined question.** If the outcome, the population or the
  prediction time point is vague, no algorithm fixes it.
- **Create information that is not in the data.** With 240 samples and 12
  measurements you cannot learn a 500-parameter relationship, whatever you fit.
- **Tell you it is being fooled.** A model trained on a leaked variable reports a
  near-perfect score with no warning of any kind. You will see this happen in
  Part 6.
- **Generalise beyond the population it was trained on**, unless you test that
  explicitly.

## 1.4 The honest summary

> A supervised model is a sophisticated way of describing the association
> structure in one dataset, packaged so it can be applied to a new sample.
> Everything that makes it *useful* &mdash; and everything that makes it *credible* &mdash;
> lives in the study design around it, not in the algorithm.

---
**CHAT CHECK-IN 1**

For each question below, post in the chat `P` (prediction) or `E` (explanation):

1. *Does higher circulating tumour DNA shedding cause worse survival?*
2. *Given a blood sample taken today, how likely is this patient to have a
   colorectal cancer detected at colonoscopy within 3 months?*
3. *Which methylation markers best separate cases from controls in my cohort?*
4. *Would lowering CRP reduce the risk of recurrence?*

Post your four letters as one line, e.g. `E P P E`.

---

> **Optional image prompt for later**
>
> *Prediction versus explanation in cancer research.* Two side-by-side panels,
> clean flat vector style on a white background, minimal text. Left panel
> "Prediction": a blood tube, an arrow into a box labelled "model", an arrow out
> to a dial labelled "estimated risk for this patient", with a small caption
> "accurate on new people". Right panel "Explanation": a directed graph with
> three labelled nodes (exposure, confounder, outcome) and arrows between them,
> caption "effect of changing the exposure". A vertical dashed line separates the
> panels. Colour-blind-safe palette (blue / orange / grey), no logos, no patient
> photographs, no numbers or performance figures, no implied clinical claim.

---
---
# PART 2 &mdash; Research question and data
### 0:15 &ndash; 0:35

Before any code: what exactly are we predicting, for whom, and from what?

## 2.1 Define the study before you touch the data

Five things. If you cannot write all five in one sentence each, the modelling is
premature.

| Element | Our worked example |
|---|---|
| **Population** | Adults aged 40&ndash;85 referred to a symptomatic colorectal clinic, before colonoscopy |
| **Outcome** | Colorectal cancer confirmed histologically at colonoscopy within 3 months (`crc`: 1 = yes, 0 = no) |
| **Prediction time point** | The moment the blood sample is drawn, **before** any colonoscopy or imaging |
| **Intended use** | Triage: rank referrals so that higher-risk patients are scoped first. **Not** a rule-out test, **not** a diagnosis. |
| **Predictors** | Only what is knowable at the prediction time point: demographics, routine bloods, and assay readouts from that one tube |

**Why the time point is the load-bearing one.** Anything recorded after that
moment is contaminated by knowledge of the outcome &mdash; even things that look
administrative. Our dataset contains `days_to_colonoscopy`, which is exactly
this: patients whose results worried someone were scoped faster. It is a real
column in a real database, it correlates beautifully with the outcome, and
putting it in a model produces a triumphant and completely worthless result.
We will do that deliberately in Part 6.

**A test to apply to every candidate predictor:** *could I have written this
value down on the day the blood was drawn, without knowing what happened next?*
If no, it is not a predictor.

## 2.2 Predictors are not outcomes, and some variables are neither

Three failure modes, all common:

- **The outcome in disguise.** A variable that is a consequence of the diagnosis
  (treatment started, referral urgency, a pathology code) is not a predictor.
- **The outcome itself, slightly re-labelled.** Easier to do than it sounds when
  a clinical database has 300 columns and four of them encode the same event.
- **A variable that will not exist at deployment.** If the model is meant to run
  on a blood draw, it cannot use a CT finding.

## 2.3 Small samples and high dimensions

This is the defining statistical problem of liquid-biopsy work, so it is worth
being precise about it.

- What matters is not *n*, but the number of **events** (people in the smaller
  class) relative to the number of **features** you let the model consider.
- A common rule of thumb for a stable logistic model is **at least 10 events per
  candidate predictor** &mdash; and "candidate" includes every feature you *looked at*
  while deciding, not just the ones that survived.
- With 800 methylation features and 60 cases, the number of ways to separate the
  classes perfectly by chance is effectively unlimited. Something will always
  look significant.
- The consequence is not usually a model that fails loudly. It is a model that
  **succeeds in your data and fails in anyone else's** &mdash; and you find out two
  years later.

Today's dataset is deliberately modest: **240 samples, 13 candidate predictors,
26% cases**. That is 63 events for 13 features &mdash; **about 4.8 events per
predictor**, already below the rule of thumb, in a dataset built to be
comfortable. Keep that number in mind when you look at how much the results move
in Part 4.

## 2.4 The dataset

Run the next cell. It builds a **synthetic** colorectal liquid-biopsy dataset in
memory &mdash; no download, no internet, same numbers for everyone.

It is built to contain the problems you will meet in real data: missing values,
class imbalance, uninformative columns, and one leaky variable.

In [ ]:
def make_colomark_data(n=240, random_state=RANDOM_STATE):
    """Create a SYNTHETIC colorectal liquid-biopsy teaching dataset.

    NOT real patients. NOT validated biomarkers. The variable names are
    realistic so the workflow feels familiar; the values are simulated.
    """
    rng = np.random.default_rng(random_state)

    # Outcome first: ~25% of this symptomatic-clinic cohort have CRC
    y = (rng.random(n) < 0.25).astype(int)

    # Realistic wrinkle: a third of cancers are "low shedders" and look like
    # controls on the assay. This is why the task is hard, not easy.
    low_shedder = np.zeros(n, dtype=bool)
    low_shedder[y == 1] = rng.random(int(y.sum())) < 0.35
    eff = np.where(y == 1, np.where(low_shedder, 0.25, 1.0), 0.0)

    df = pd.DataFrame({
        "patient_id":            [f"CM{i:04d}" for i in range(1, n + 1)],
        # --- demographics, known at the blood draw
        "age":                   np.clip(rng.normal(62, 10, n) + 3 * y, 30, 90).round(0),
        "sex":                   rng.choice(["F", "M"], n, p=[0.47, 0.53]),
        "bmi":                   np.clip(rng.normal(26.5, 4.0, n), 16, 45).round(1),
        # --- liquid-biopsy assay readouts
        "ctdna_vaf":             np.exp(rng.normal(-6.2 + 1.40 * eff, 1.0, n)).round(5),
        "cfdna_conc_ng_ml":      np.exp(rng.normal(1.9 + 0.35 * eff, 0.5, n)).round(1),
        "methyl_score":          (rng.normal(0, 1, n) + 0.85 * eff).round(3),
        "fragment_ratio":        (rng.normal(0.52, 0.045, n) - 0.021 * eff).round(3),
        # --- routine bloods
        "cea_ng_ml":             np.exp(rng.normal(0.55 + 0.65 * eff, 0.65, n)).round(2),
        "platelet_count":        np.clip(rng.normal(255, 62, n) + 16 * eff, 90, 600).round(0),
        "crp_mg_l":              np.exp(rng.normal(0.65 + 0.38 * eff, 0.9, n)).round(2),
        # --- technical columns that carry no signal at all
        "assay_qc_score":        rng.normal(0, 1, n).round(3),
        "sample_storage_months": np.abs(rng.normal(0, 1, n) * 6 + 9).round(1),
        "lab_tech_index":        rng.normal(0, 1, n).round(3),
        # --- RECORDED AFTER THE PREDICTION TIME POINT. Not a predictor.
        "days_to_colonoscopy":   np.clip(rng.normal(np.where(y == 1, 13, 44),
                                                    np.where(y == 1, 6, 15), n), 2, 120).round(0),
        # --- the outcome
        "crc":                   y,
    })

    # Missing values, as in any real clinical dataset
    df.loc[rng.random(n) < 0.18, "cea_ng_ml"] = np.nan
    df.loc[rng.random(n) < 0.11, "crp_mg_l"] = np.nan
    return df


data = make_colomark_data()

print("SYNTHETIC DATA - no real patients, no validated biomarkers.")
print("rows (patients), columns:", data.shape)
data.head()

Now look at what is actually in the table: how many of each class, what type
each column is, and where the holes are.

In [ ]:
# TODO: change show_percent from False to True and re-run
show_percent = False

print("=== Outcome balance (crc: 1 = cancer, 0 = no cancer) ===")
if show_percent:
    print(data["crc"].value_counts(normalize=True).round(3))
else:
    print(data["crc"].value_counts())

n_cases = int(data["crc"].sum())
print(f"\ncases: {n_cases}   controls: {len(data) - n_cases}   prevalence: {data['crc'].mean():.1%}")

print("\n=== Column types ===")
print(data.dtypes)

print("\n=== Missing values per column ===")
missing = data.isna().sum()
print(missing[missing > 0])

---
**CHAT CHECK-IN 2**

Run the cell above, then post one line in the chat:

`n = <rows>, cases = <number>, missing CEA = <number>`

---

---
**ACTIVITY 1 &mdash; 3 minutes, in the chat**

Take **your own** project and write the five lines:

```
Population:
Outcome:
Prediction time point:
Intended use:
Predictors available at that time point:
```

Post the two you find hardest to fill in. If you cannot name a prediction time
point, say so &mdash; that is a common and informative answer, and it usually means
your question is explanatory rather than predictive.

---

> **Optional image prompt for later**
>
> *A patient-level data table showing outcome, predictors, and prediction time
> point.* Clean flat vector illustration, white background, minimal text. A
> simple table with four or five rows, each row a patient. Columns on the left
> shaded blue and labelled "predictors (known at blood draw)"; one column on the
> right shaded orange and labelled "outcome (known later)". A vertical dashed
> line between them labelled "prediction time point", with a small clock icon.
> One greyed-out column to the right of the dashed line labelled "recorded after
> &mdash; not a predictor". Colour-blind-safe palette, no real patient data, no
> identifiable information, no clinical claims.

---
---
# PART 3 &mdash; Hands-on workflow
### 0:35 &ndash; 1:10

Load, inspect, split, impute, fit. Five steps, in that order, every time.

## 3.1 Step 1 &mdash; Inspect before you model

Two minutes of looking saves two weeks of debugging. What you are checking:

- Is the **outcome** what you think it is, and coded how you think?
- What **type** is each column &mdash; numeric, categorical, identifier?
- How **imbalanced** are the classes?
- Where are the **missing values**, and are they missing for a reason?
- Is anything on a wildly different scale, or obviously degenerate?

In [ ]:
print("=== Summary of the numeric columns ===")
display(data.describe().T.round(3))

print("\n=== Case vs control, a few key markers ===")
display(
    data.groupby("crc")[["ctdna_vaf", "methyl_score", "cea_ng_ml", "days_to_colonoscopy"]]
        .median()
        .round(4)
)

Look at the last column of that table.

`days_to_colonoscopy` separates the groups more cleanly than any assay readout.
That is not a discovery &mdash; it is the care pathway reacting to information the
model is not supposed to have. Hold that thought until Part 6.

Now a picture of the signal we *are* allowed to use.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

for ax, col, title in zip(
    axes,
    ["ctdna_vaf", "methyl_score"],
    ["ctDNA VAF (log scale)", "Methylation score"],
):
    for label, name in [(0, "no cancer"), (1, "cancer")]:
        vals = data.loc[data["crc"] == label, col]
        ax.hist(vals, bins=25, alpha=0.6, label=name)
    ax.set_title(title)
    ax.set_xlabel(col)
    if col == "ctdna_vaf":
        ax.set_xscale("log")

axes[0].set_ylabel("number of patients")
axes[0].legend()
plt.tight_layout()
plt.show()

print("Note the overlap. This is a realistic amount of separation, not a clean one.")

---
**CHAT CHECK-IN 3**

Look at the two histograms. Post in the chat, one line:

`separation: good / partial / poor` &mdash; and in three words, why.

---

## 3.2 Step 2 &mdash; Separate predictors from the outcome

Two rules, both easy to break by accident:

1. The outcome column (`crc`) **never** appears among the predictors.
2. Identifiers (`patient_id`) and post-outcome variables (`days_to_colonoscopy`)
   are excluded too &mdash; the first carries no information, the second carries the
   answer.

We list the predictors explicitly rather than writing "everything except `crc`".
Explicit lists are how you avoid the leak you did not think of.

In [ ]:
NUMERIC_FEATURES = [
    "age", "bmi",
    "ctdna_vaf", "cfdna_conc_ng_ml", "methyl_score", "fragment_ratio",
    "cea_ng_ml", "platelet_count", "crp_mg_l",
    "assay_qc_score", "sample_storage_months", "lab_tech_index",
]
CATEGORICAL_FEATURES = ["sex"]
TARGET = "crc"

# Deliberately NOT included:
#   patient_id          -> identifier, no information
#   days_to_colonoscopy -> recorded AFTER the prediction time point (Part 6)

X = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = data[TARGET]

assert TARGET not in X.columns, "The outcome must never be a predictor."
assert "days_to_colonoscopy" not in X.columns

print("feature matrix X:", X.shape, " (patients, features)")
print("outcome vector y:", y.shape)
print("candidate predictors:", X.shape[1])
print(f"events per candidate predictor: {y.sum() / X.shape[1]:.1f}")

## 3.3 Step 3 &mdash; The train/test split

We hold back part of the data, train on the rest, and use the held-back part
**once**, at the end, to estimate how the model behaves on people it has never
seen.

Three choices to make explicitly:

- `test_size=0.30` &mdash; a compromise. Larger test set, more reliable estimate, less
  data to learn from. With 240 patients there is no comfortable answer.
- `stratify=y` &mdash; keeps the same case proportion in both halves. With 26% cases
  and a small test set, an unstratified split can easily hand you a test set with
  12 cases in it by chance.
- `random_state=42` &mdash; reproducibility. Note what this does **not** mean: it does
  not mean the result is stable, only that it is repeatable. Part 4 is about
  exactly that difference.

> **The rule for the rest of the session:** the test set is sealed. We do not
> look at it, select features with it, tune a threshold on it, or choose a model
> with it. Every one of those turns an honest estimate into an optimistic one.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,                 # preserve the case/control ratio in both halves
    random_state=RANDOM_STATE,  # reproducible, NOT stable - see Part 4
)

print(f"training set: {X_train.shape[0]:3d} patients, {int(y_train.sum()):2d} cases "
      f"({y_train.mean():.1%})")
print(f"test set    : {X_test.shape[0]:3d} patients, {int(y_test.sum()):2d} cases "
      f"({y_test.mean():.1%})")
print()
print("The test set holds", int(y_test.sum()), "cases. Every performance number we")
print("report at the end rests on that many events.")

---
**CHAT CHECK-IN 4**

Post in the chat: `test n = <number>, test cases = <number>`

Then answer in one word: with that many cases in the test set, would you report
sensitivity to two decimal places?

---

## 3.4 Step 4 &mdash; Missing values, done correctly

Two columns have gaps: `cea_ng_ml` (40 missing) and `crp_mg_l` (25). Most models
refuse to run with missing values, so you must decide what to do.

The simple option is **imputation**: fill each gap with a summary of that column
&mdash; here, the median.

The question that matters is not *which* value, but **which data the value is
computed from**.

| | What happens | Verdict |
|---|---|---|
| Median of the **whole dataset**, then split | The training data has absorbed information from the test patients | **Leakage.** Mild here, severe with scaling or feature selection |
| Median of the **training set only**, applied to both | The test set stays genuinely unseen | **Correct** |

The same applies to scaling (the mean and SD must come from the training data),
to feature selection, and to anything else that *learns* a number from the data.

**A scikit-learn `Pipeline` is how you stop having to remember this.** It chains
the preprocessing and the model into one object. When you call `.fit()`, every
step learns from the training data only; when you call `.predict()`, the stored
training values are applied. Inside cross-validation, it re-learns them in each
fold automatically. This is the single most useful habit in this notebook.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# --- numeric columns: fill gaps with the TRAINING median, then standardise
numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])

# --- categorical columns: one column per category
categorical_steps = OneHotEncoder(handle_unknown="ignore", drop="if_binary")

preprocess = ColumnTransformer([
    ("num", numeric_steps,      NUMERIC_FEATURES),
    ("cat", categorical_steps,  CATEGORICAL_FEATURES),
])

baseline_model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

baseline_model

Now fit it. One line. Everything above was the thinking; this is the easy part.

In [ ]:
baseline_model.fit(X_train, y_train)

# What the imputer learned - from the training data only
learned_medians = baseline_model.named_steps["preprocess"] \
                                .named_transformers_["num"] \
                                .named_steps["impute"].statistics_

print("Median learned for cea_ng_ml (training set only):",
      round(float(learned_medians[NUMERIC_FEATURES.index("cea_ng_ml")]), 3))
print("Median of cea_ng_ml across the WHOLE dataset     :",
      round(float(data["cea_ng_ml"].median()), 3))
print()
print("Different numbers. Using the second one would be leakage.")

## 3.5 Step 5 &mdash; A first look at performance (and why it is not enough)

`.score()` on a classifier gives **accuracy**: the proportion of test patients
put in the right box.

Before you read it, work out what accuracy a model that has learned nothing
would get. In a cohort that is 74% controls, "always predict no cancer" scores
0.74 without looking at a single blood sample. That is the number any real model
has to beat &mdash; and the bar is much higher than zero.

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print(f"'always predict no cancer' accuracy : {dummy.score(X_test, y_test):.3f}")
print(f"logistic regression accuracy        : {baseline_model.score(X_test, y_test):.3f}")
print()
print("Hold that comparison in mind. We unpack it properly in Part 5.")

> **Optional image prompt for later**
>
> *Train/test split and cross-validation.* Clean flat vector diagram, white
> background, minimal text. Top row: a horizontal bar of 10 equal segments, 7
> shaded blue labelled "training", 3 shaded orange labelled "test (sealed until
> the end)". Below it, a five-row stack showing 5-fold cross-validation *within
> the blue portion only*: in each row one segment is highlighted as "validation"
> and the rest as "fit", with the orange test portion greyed out and untouched
> across all five rows. A small padlock icon on the orange block. Colour-blind-
> safe palette, no numbers, no performance figures, no clinical imagery.

> **Optional image prompt for later**
>
> *Missing-value imputation performed within a training pipeline.* Clean flat
> vector diagram, white background, minimal text. A small data table on the left
> with two visibly empty cells. An arrow into a rounded box labelled "pipeline"
> containing three stacked stages: "impute (median learned here)", "scale",
> "fit model". A dashed arrow labelled "learned values applied, not recomputed"
> running from the pipeline box to a separate, clearly-marked "test data" table on
> the right. A red crossed-out arrow running directly from the test table back
> into the "impute" stage, labelled "never". Colour-blind-safe palette, no
> clinical claims.

---
---
# PART 4 &mdash; Small exercise
### 1:10 &ndash; 1:20

Ten minutes, working on your own. Cameras off if you like.

## Your task

The model above was evaluated on **one** train/test split, chosen by
`random_state=42`. Nothing about that split is special.

The cell below re-runs the whole workflow &mdash; split, impute, scale, fit, evaluate
&mdash; for several different splits, and collects the test ROC-AUC each time.

1. Run it as given (`n_splits = 3`).
2. Change `n_splits` to **10** and run it again.
3. Look at the **range** between the lowest and highest AUC.

**Expected output:** a table of AUCs, their mean, and the min&ndash;max spread.

**Then answer, in the chat:** if you had run this analysis once and reported the
best number, how far from the truth could you have been?

In [ ]:
from sklearn.metrics import roc_auc_score

# TODO: change n_splits from 3 to 10, then re-run this cell
n_splits = 3

results = []
for seed in range(1, n_splits + 1):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=seed
    )

    # A FRESH pipeline each time - nothing carries over between splits
    model = Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale",  StandardScaler())]), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"),
                    CATEGORICAL_FEATURES),
        ])),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    model.fit(X_tr, y_tr)

    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    acc = model.score(X_te, y_te)
    results.append({"split_seed": seed, "test_auc": round(auc, 3), "test_accuracy": round(acc, 3)})

results = pd.DataFrame(results)
display(results)

print(f"splits run : {n_splits}")
print(f"mean AUC   : {results['test_auc'].mean():.3f}")
print(f"lowest AUC : {results['test_auc'].min():.3f}")
print(f"highest AUC: {results['test_auc'].max():.3f}")
print(f"SPREAD     : {results['test_auc'].max() - results['test_auc'].min():.3f}")

---
**Write your answer here** (double-click this cell to edit it):

- Lowest AUC I got: 
- Highest AUC I got: 
- If I reported only the best split, I would be overstating performance by: 

---

---
---
# PART 5 &mdash; Model evaluation
### 1:20 &ndash; 1:50

One number is never enough. Here is what to report instead, and why.

## 5.1 State the positive class first

Every metric below depends on which class is "positive". Get this wrong in a
paper and every number in the table is wrong.

**Here: positive = `crc` = 1 = colorectal cancer confirmed at colonoscopy.**

A **confusion matrix** is just the four ways a prediction can land:

| | predicted: no cancer | predicted: cancer |
|---|---|---|
| **truly no cancer** | true negative (TN) | false positive (FP) |
| **truly cancer** | **false negative (FN)** | true positive (TP) |

In this clinical setting the four cells have wildly different costs. A false
positive is an unnecessary colonoscopy. A false negative is a missed cancer.
Accuracy weights them identically, which is the first reason it misleads.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = baseline_model.predict(X_test)                  # hard 0/1 labels
y_prob = baseline_model.predict_proba(X_test)[:, 1]      # probability of class 1

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(4.2, 3.8))
ConfusionMatrixDisplay(cm, display_labels=["no cancer", "cancer"]).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title("Test set, threshold 0.5")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"true negatives  TN = {tn:3d}   (correctly left alone)")
print(f"false positives FP = {fp:3d}   (unnecessary colonoscopy)")
print(f"false negatives FN = {fn:3d}   (MISSED CANCER)")
print(f"true positives  TP = {tp:3d}   (cancer correctly flagged)")

---
**CHAT CHECK-IN 5**

Before scrolling further &mdash; read the matrix and post in the chat:

`of the cancers in the test set, we caught <TP> and missed <FN>`

---

## 5.2 Sensitivity, specificity, and the two that depend on prevalence

$$\text{sensitivity (recall)} = \frac{TP}{TP + FN} \qquad
\text{specificity} = \frac{TN}{TN + FP}$$

- **Sensitivity** &mdash; of the people who have cancer, what fraction did we flag?
- **Specificity** &mdash; of the people who do not, what fraction did we leave alone?

Both are properties of the model *at one threshold*, and neither depends on how
common the disease is.

$$\text{PPV} = \frac{TP}{TP + FP} \qquad \text{NPV} = \frac{TN}{TN + FN}$$

- **PPV** &mdash; if the model says cancer, how often is it right?
- **NPV** &mdash; if the model says no cancer, how often is it right?

**These two depend heavily on prevalence.** Our cohort is 26% cases because it is
a symptomatic referral population. Move the same model to population screening at
0.3% prevalence and the PPV collapses, with no change to the model at all. This
is the single most common reason a promising biomarker study does not survive
contact with a screening setting.

In [ ]:
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp)
npv         = tn / (tn + fn)
accuracy    = (tp + tn) / cm.sum()

print(f"accuracy    {accuracy:.3f}")
print(f"sensitivity {sensitivity:.3f}   <- of {tp + fn} cancers, {tp} flagged")
print(f"specificity {specificity:.3f}")
print(f"PPV         {ppv:.3f}   (at this cohort's {y_test.mean():.0%} prevalence)")
print(f"NPV         {npv:.3f}")

# What the same sensitivity/specificity give at screening prevalence
for prevalence in [0.26, 0.05, 0.003]:
    ppv_at = (sensitivity * prevalence) / (
        sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
    )
    print(f"\nat prevalence {prevalence:>6.1%}: PPV = {ppv_at:.3f}")

## 5.3 Probabilities, thresholds, and the ROC curve

The model does not really output 0 or 1. It outputs a **probability**, and
`.predict()` applies a threshold of 0.5 to it.

That 0.5 is a default, not a property of the model, and it is almost never the
right choice clinically. Lower it and you catch more cancers and order more
unnecessary colonoscopies; raise it and the reverse. **The threshold encodes how
you trade a missed cancer against an unnecessary procedure** &mdash; which is a
clinical decision, not a statistical one.

The **ROC curve** shows every threshold at once: sensitivity on the y-axis
against 1 &minus; specificity on the x-axis. **ROC-AUC** is the area under it, and
has one clean interpretation:

> Take one random patient with cancer and one without. The AUC is the probability
> the model assigns the higher risk to the one with cancer.

0.5 is a coin flip. 1.0 is perfect separation. **AUC is threshold-free**, which is
its strength (it summarises ranking) and its weakness (it says nothing about
whether any usable threshold exists, and nothing about calibration).

In [ ]:
from sklearn.metrics import roc_curve, RocCurveDisplay

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(4.6, 4.2))
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc).plot(ax=ax, name="logistic regression")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="no better than chance")
ax.scatter(1 - specificity, sensitivity, s=70, zorder=5, color="crimson",
           label="threshold = 0.5")
ax.set_title("ROC curve, test set")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"ROC-AUC = {auc:.3f}")
print()
print("What a different threshold would buy us:")
for t in [0.50, 0.30, 0.20]:
    pred_t = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    print(f"  threshold {t:.2f}:  sensitivity {tp_t/(tp_t+fn_t):.2f}   "
          f"specificity {tn_t/(tn_t+fp_t):.2f}   "
          f"missed cancers {fn_t:2d}   unnecessary scopes {fp_t:2d}")

## 5.4 Why accuracy alone misleads

Put the numbers side by side.

In [ ]:
print(f"{'':34s}{'accuracy':>10s}{'sensitivity':>13s}{'specificity':>13s}")
print("-" * 70)

dummy_pred = dummy.predict(X_test)
cm_d = confusion_matrix(y_test, dummy_pred, labels=[0, 1])
tn_d, fp_d, fn_d, tp_d = cm_d.ravel()
print(f"{'always predict no cancer':34s}"
      f"{(tp_d+tn_d)/cm_d.sum():>10.3f}"
      f"{tp_d/(tp_d+fn_d):>13.3f}"
      f"{tn_d/(tn_d+fp_d):>13.3f}")
print(f"{'logistic regression (t=0.5)':34s}"
      f"{accuracy:>10.3f}{sensitivity:>13.3f}{specificity:>13.3f}")

print()
print("The useless model catches ZERO cancers and still scores 0.736, because")
print("74% of this cohort is cancer-free. Ten points of accuracy separate it from")
print("a model that misses 11 of 19 cancers. Accuracy is measuring the wrong thing.")

**What to report instead.** For a binary clinical classifier, the minimum
honest set is:

- the **confusion matrix** itself, with the positive class stated
- **sensitivity and specificity** at the threshold you are proposing, *with the
  threshold named*
- **ROC-AUC** as the threshold-free summary of ranking
- **PPV and NPV** with the prevalence they assume stated explicitly
- an **interval** around each of these
- the **comparison model** &mdash; a majority-class baseline at minimum, and ideally
  the existing clinical standard. A model that does not beat the current practice
  is not a result, however good its AUC looks.

**And one thing none of them establishes:** clinical utility. Sensitivity,
specificity and AUC describe discrimination on a retrospective dataset. Whether a
model improves patient outcomes is a question about calibration, decision
thresholds, workflow and cost &mdash; answered by decision-curve analysis and
ultimately a prospective study, not by any number computed in this notebook.

## 5.5 How uncertain is that AUC?

Our test set has 72 patients and 19 cancers. Every number above is an estimate
from that handful of events.

A **bootstrap** gives a quick sense of the uncertainty: resample the test set
with replacement many times, recompute the AUC each time, and look at the spread.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
boot_aucs = []

y_test_arr = y_test.to_numpy()
for _ in range(2000):
    idx = rng.integers(0, len(y_test_arr), len(y_test_arr))
    if len(np.unique(y_test_arr[idx])) < 2:     # need both classes present
        continue
    boot_aucs.append(roc_auc_score(y_test_arr[idx], y_prob[idx]))

boot_aucs = np.array(boot_aucs)
lo, hi = np.percentile(boot_aucs, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(5.2, 3))
ax.hist(boot_aucs, bins=40, color="steelblue", alpha=0.8)
ax.axvline(auc, color="crimson", lw=2, label=f"point estimate {auc:.3f}")
ax.axvline(lo, color="k", ls="--", lw=1)
ax.axvline(hi, color="k", ls="--", lw=1, label=f"95% CI {lo:.2f}-{hi:.2f}")
ax.set_xlabel("bootstrap ROC-AUC")
ax.set_ylabel("count")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"ROC-AUC {auc:.3f}, 95% bootstrap CI [{lo:.3f}, {hi:.3f}]")
print(f"interval width: {hi - lo:.3f}")

---
**CHAT CHECK-IN 6**

Post your interval: `AUC = <point>, 95% CI <low> to <high>`

Then: a competing assay reports AUC 0.86 in a different cohort. On this evidence,
is our model better, worse, or indistinguishable?

---

> **Optional image prompt for later**
>
> *Confusion matrix with sensitivity and specificity.* Clean flat vector diagram,
> white background, minimal text. A 2&times;2 grid with rows labelled "truly
> cancer" / "truly no cancer" and columns labelled "predicted cancer" /
> "predicted no cancer". The two correct cells shaded calm blue, the two error
> cells shaded muted orange, with the false-negative cell outlined and annotated
> "missed cancer". To the right, two small callouts: one bracketing the top row
> labelled "sensitivity = TP / row total", one bracketing the bottom row labelled
> "specificity = TN / row total". Colour-blind-safe palette, no real patient data,
> no performance numbers, no diagnostic claim.

---
---
# PART 6 &mdash; Common pitfalls
### 1:50 &ndash; 2:15

The failure modes that produce published, peer-reviewed, irreproducible results.

## 6.1 Data leakage

> **Leakage** is any situation where the model has access, during training, to
> information it would not have at the moment of prediction.

The signature is always the same: **performance that is too good, and no error
message.** Nothing crashes. Nothing warns you. The model simply reports a
spectacular result that evaporates on new data.

Two forms, and we will do both.

### Form A &mdash; a variable recorded after the outcome was known

`days_to_colonoscopy` is in the dataset. It is real, it is in the database, and
it is recorded after someone decided how urgent the referral was. Put it in the
model and watch.

In [ ]:
LEAKY_NUMERIC = NUMERIC_FEATURES + ["days_to_colonoscopy"]   # the mistake

X_leaky = data[LEAKY_NUMERIC + CATEGORICAL_FEATURES]
X_tr_lk, X_te_lk, y_tr_lk, y_te_lk = train_test_split(
    X_leaky, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

leaky_model = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale",  StandardScaler())]), LEAKY_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"),
                CATEGORICAL_FEATURES),
    ])),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
leaky_model.fit(X_tr_lk, y_tr_lk)
leaky_auc = roc_auc_score(y_te_lk, leaky_model.predict_proba(X_te_lk)[:, 1])

print(f"honest model (no leak)        ROC-AUC = {auc:.3f}")
print(f"with days_to_colonoscopy      ROC-AUC = {leaky_auc:.3f}   <- publishable-looking")
print()
print("Note it is not a suspicious 1.000 - it is a plausible, excellent 0.97.")
print("That is what makes leakage hard to spot: it looks like a good result,")
print("not like a bug.")
print()
print("No warning. No error. The code is correct. The STUDY is wrong:")
print("at the moment of the blood draw, days_to_colonoscopy does not yet exist.")

---
**CHAT CHECK-IN 7**

Post one variable from **your own** data that fails the test:

> *Could I have written this value down on the day the sample was taken, without
> knowing what happened next?*

If you cannot think of one, post the variable you are least sure about.

---

### Form B &mdash; choosing features before splitting

This is subtler, far more common, and much more damaging.

The tempting workflow, which appears in a great deal of published
high-dimensional biomarker work:

1. Measure 4,000 features on 120 samples
2. Correlate each feature with the outcome **across all the data**
3. Keep the 20 best
4. Cross-validate a model on those 20
5. Report a beautiful AUC

Step 2 uses the outcome of **every** sample, including the ones later used for
validation. The selection has already seen the answers.

To show exactly how badly this fails, the next cell uses data with **no signal at
all** &mdash; 4,000 pure random-noise features and a coin-flip outcome. The true AUC
is 0.5 by construction. Any value above 0.5 is manufactured by the procedure.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif

noise_rng = np.random.default_rng(0)
X_noise = noise_rng.normal(size=(120, 4000))      # 4000 features of pure noise
y_noise = noise_rng.integers(0, 2, 120)           # a coin flip
# TRUE AUC = 0.500, by construction. There is nothing to find.

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# --- THE WRONG WAY: select features using ALL the data, then cross-validate
corr = np.abs([np.corrcoef(X_noise[:, j], y_noise)[0, 1] for j in range(X_noise.shape[1])])
top20 = np.argsort(corr)[-20:]
auc_wrong = cross_val_score(
    LogisticRegression(max_iter=1000), X_noise[:, top20], y_noise,
    cv=cv, scoring="roc_auc"
).mean()

# --- THE RIGHT WAY: selection INSIDE the pipeline, re-done in every fold
correct_pipeline = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("clf",    LogisticRegression(max_iter=1000)),
])
auc_right = cross_val_score(
    correct_pipeline, X_noise, y_noise, cv=cv, scoring="roc_auc"
).mean()

print("Data contains NO signal whatsoever. True AUC = 0.500")
print("-" * 58)
print(f"selection before splitting (WRONG)   CV AUC = {auc_wrong:.3f}")
print(f"selection inside the pipeline (RIGHT) CV AUC = {auc_right:.3f}")
print()
print("The wrong version manufactures a strong result out of random numbers.")

## 6.2 Class imbalance

Our cohort is 26% cases. Two consequences, both already visible:

- **Accuracy becomes meaningless** &mdash; the majority-class baseline scored 0.736.
- **The 0.5 threshold drifts towards the majority class**, which is precisely why
  our sensitivity was 0.42 while specificity was 0.98.

What actually helps:

- **Stratify** the split and the cross-validation folds (we did)
- **Report metrics that are not dominated by the majority class** &mdash; sensitivity,
  specificity, ROC-AUC, and precision-recall AUC when cases are rare
- **Move the threshold**, chosen on training/CV data, to match the clinical cost
- **`class_weight="balanced"`**, which tells the model to penalise errors on the
  rare class more heavily

What usually does not help as much as people hope: synthetic oversampling (SMOTE
and relatives). It is widely used, frequently applied *before* the split (which is
leakage), and rarely outperforms simply moving the threshold.

In [ ]:
balanced_model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced",
                                      random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

cm_b = confusion_matrix(y_test, balanced_model.predict(X_test), labels=[0, 1])
tn_b, fp_b, fn_b, tp_b = cm_b.ravel()

print(f"{'':24s}{'accuracy':>10s}{'sens':>8s}{'spec':>8s}{'missed':>8s}{'AUC':>8s}")
print("-" * 66)
print(f"{'default':24s}{accuracy:>10.3f}{sensitivity:>8.2f}{specificity:>8.2f}"
      f"{fn:>8d}{auc:>8.3f}")
print(f"{'class_weight=balanced':24s}"
      f"{(tp_b+tn_b)/cm_b.sum():>10.3f}{tp_b/(tp_b+fn_b):>8.2f}{tn_b/(tn_b+fp_b):>8.2f}"
      f"{fn_b:>8d}"
      f"{roc_auc_score(y_test, balanced_model.predict_proba(X_test)[:, 1]):>8.3f}")
print()
print("Accuracy went DOWN, missed cancers went DOWN, AUC barely moved.")
print("Which of those three numbers matters depends on the intended use.")

## 6.3 Overfitting

A model overfits when it learns the noise in the training data as if it were
signal. It shows up as a **large gap between training and test performance**.

The comparison below uses an unconstrained decision tree &mdash; a model with enough
flexibility to memorise 168 patients exactly.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),   # no depth limit
]).fit(X_train, y_train)

rows = []
for name, mdl in [("logistic regression", baseline_model),
                  ("unconstrained decision tree", tree_model)]:
    rows.append({
        "model":     name,
        "train AUC": round(roc_auc_score(y_train, mdl.predict_proba(X_train)[:, 1]), 3),
        "test AUC":  round(roc_auc_score(y_test,  mdl.predict_proba(X_test)[:, 1]), 3),
    })

gap_table = pd.DataFrame(rows)
gap_table["gap"] = (gap_table["train AUC"] - gap_table["test AUC"]).round(3)
display(gap_table)

print("A perfect training score is not an achievement. It is a symptom.")
print("Always report both numbers - a paper that shows only one is hiding the gap.")

## 6.4 Cross-validation, and what the test set is actually for

A single split gives one estimate from one small test set. You measured how
unstable that is in Part 4.

**Cross-validation** uses the training data more efficiently: split it into *k*
folds, train on *k*&minus;1, validate on the held-out one, rotate, average.

The division of labour that keeps you honest:

| | Used for | Touched how often |
|---|---|---|
| **Cross-validation** (on training data) | Comparing models, tuning, feature selection, choosing a threshold | As often as you like |
| **Test set** | One final, unbiased estimate | **Once**, at the very end |

The moment you choose anything based on the test set, it stops being a test set
and becomes part of your training procedure &mdash; and your reported performance
becomes optimistic by an unknown amount.

In [ ]:
cv_scores = cross_val_score(
    baseline_model, X_train, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
)

print("5-fold CV on the TRAINING data only")
print("fold AUCs:", np.round(cv_scores, 3))
print(f"mean {cv_scores.mean():.3f}  (SD {cv_scores.std():.3f})")
print()
print(f"single held-out test AUC: {auc:.3f}")
print()
print("The means are close. Look at the FOLD-TO-FOLD RANGE instead: the same")
print("model, the same data, scores very differently depending on which ~34")
print("patients happen to land in the validation fold. No single number here is")
print("'the' performance - which is the honest summary of a dataset this size.")

## 6.5 External validation and distribution shift

Everything above is **internal** validation &mdash; one cohort, split in two. It
answers: *does this model work on patients like the ones I already have?*

It does not answer: *does this model work anywhere else?* That requires an
**external** cohort &mdash; a different site, a different time period, a different
assay run, ideally a different population.

Performance almost always drops. The usual causes:

- **Case mix** differs &mdash; a different disease spectrum or stage distribution
- **Prevalence** differs &mdash; PPV and NPV move even when the model does not
- **Assay and pre-analytics** differ &mdash; collection tubes, processing delay,
  sequencing depth, batch
- **The care pathway** differs &mdash; which changes what the recorded variables mean

> **State this plainly in your own papers:** a single train/test split, as used
> throughout this notebook, is a teaching simplification. It is not a clinical
> validation. A model intended for clinical use requires, at minimum, external
> validation in an independent cohort, assessment of calibration (not just
> discrimination), and a prospective evaluation of whether using it changes
> patient outcomes.

> **Optional image prompt for later**
>
> *Data leakage &mdash; information from the future entering the training data.*
> Clean flat vector diagram, white background, minimal text. A horizontal
> timeline with two marked points: "blood draw (prediction time)" on the left and
> "colonoscopy result (outcome known)" on the right. A rounded box labelled
> "model training data" sits under the left point. A curved red dashed arrow runs
> backwards from the right point into the training box, crossing the timeline,
> labelled "leaked variable". A small warning triangle where the arrow crosses.
> Colour-blind-safe palette, no patient imagery, no performance numbers.

> **Optional image prompt for later**
>
> *External validation across hospitals or cohorts.* Clean flat vector diagram,
> white background, minimal text. On the left, a building icon labelled
> "development cohort, site A" with a small box labelled "model" beneath it. An
> arrow crosses to the right to two further building icons labelled "site B" and
> "site C". Under each, a small simple gauge, with site A's needle highest and
> sites B and C visibly lower, labelled "performance usually drops". A short
> caption strip underneath reads "different case mix, prevalence, assay".
> Colour-blind-safe palette, generic unbranded buildings, no numbers, no real
> institutions, no clinical claims.

---
---
# PART 7 &mdash; Group exercise and discussion
### 2:15 &ndash; 2:30

Read the study below as if you were reviewing it.

## The study

> **"A multi-marker cfDNA signature detects colorectal cancer with high accuracy"**
>
> We assembled **94 patients** with histologically confirmed colorectal cancer
> from our institutional biobank and **96 healthy controls** recruited from a
> staff blood-donation drive. Cancer samples were collected at the oncology day
> unit between 2019 and 2022; control samples were collected in 2023 at the
> occupational health clinic.
>
> Plasma cfDNA was extracted and **1,200 methylation features** were quantified.
> Cases and controls were processed in **separate batches**, each with its own
> extraction kit lot.
>
> Features were ranked by t-test p-value **across the full dataset** and the top
> **40** retained. A random forest was trained on these 40 features using
> **10-fold cross-validation**, achieving **AUC 0.97 (accuracy 94%)**. Model
> hyperparameters were selected to **maximise cross-validated AUC**.
>
> The signature includes several genes with **known roles in colorectal
> tumourigenesis**, supporting its biological plausibility. We conclude that this
> cfDNA signature enables **accurate, non-invasive early detection of colorectal
> cancer** and is ready for evaluation as a **population screening tool**.

## Your task &mdash; in breakout groups (8 minutes)

1. List the **three most serious problems**, ranked. What would you fix first?
2. Name **one piece of evidence** you would require before believing the headline
   claim.
3. Is the AUC of 0.97 *wrong*, exactly? If not, what is it actually measuring?

Use these prompts if you get stuck: **cohort selection &middot; outcome definition
&middot; assay and batch &middot; prediction time point &middot; leakage &middot;
validation &middot; metrics &middot; intended use.**

In [ ]:
# Space for your group's notes. Edit and run, or just type - nothing depends on this.
group_notes = """
Problem 1 (most serious):
Problem 2:
Problem 3:

Evidence I would require:

Is AUC 0.97 wrong? What is it measuring?
"""
print(group_notes)

---
---
# Wrapping up

## The checklist

Keep this. It applies to your own work and to anything you review.

**Design**
- [ ] Population, outcome, prediction time point, intended use, predictors &mdash; all written down before modelling
- [ ] Every predictor passes: *could I have recorded this on the day the sample was taken?*
- [ ] Cases and controls drawn from the same population and the same pathway
- [ ] Batch, site and collection date randomised across groups, or at minimum reported and checked

**Data**
- [ ] Class balance reported
- [ ] Missing values reported, and imputation fitted on training data only
- [ ] Events per candidate predictor computed and reported

**Modelling**
- [ ] Preprocessing and feature selection inside the resampling loop (use a `Pipeline`)
- [ ] Test set used exactly once, at the end
- [ ] Both training and test performance reported

**Reporting**
- [ ] Positive class stated
- [ ] Confusion matrix, sensitivity, specificity at a named threshold
- [ ] ROC-AUC with a confidence interval
- [ ] PPV/NPV with the assumed prevalence stated
- [ ] Comparison against a baseline and against current clinical practice
- [ ] External validation, or an explicit statement that there is none
- [ ] No claim of clinical utility from discrimination metrics alone

## Where to go next

- **TRIPOD+AI** (Collins et al., *BMJ* 2024) &mdash; the reporting guideline for
  prediction-model studies using machine learning. The checklist most reviewers
  will apply to your paper. Read it before you write, not after.
- **PROBAST** &mdash; the companion risk-of-bias tool. Useful as a self-audit.
- **scikit-learn user guide**, sections on model selection and on common pitfalls
  (`scikit-learn.org/stable/common_pitfalls.html`) &mdash; short, practical, and
  written by the people who built the tools you used today.
- *An Introduction to Statistical Learning* (James, Witten, Hastie, Tibshirani) &mdash;
  freely available, and the Python edition covers everything in this session
  properly.

## One sentence to take away

> The algorithm was the easiest part of today. Everything that decided whether
> the result meant anything happened before `.fit()` was called.

---

*Questions after the session: maryam.abbasi@esg.ipsantarem.pt*

*All data in this notebook is synthetic. No part of it constitutes clinical
evidence or supports any diagnostic or treatment decision.*